In [1]:
import pandas as pd
import numpy as np

In [2]:
# Load DataSets
df = pd.read_csv("C:/Users/dell/OneDrive//Desktop/Machine Learning/NLP Project/train.txt",sep=';',header=None,names=["Text","Emotions"])
df.head()

,Text,Emotions
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


In [3]:
df.shape

(16000, 2)

In [4]:
# Let's Perform Nlp

# Step1:- Emotions value must be 0,1
df['Emotions'].unique()

unique_emotions = df['Emotions'].unique()
new_emotions = {}
i=0
for emo in unique_emotions:
    new_emotions[emo] = i
    i += 1

df['Emotions'] = df['Emotions'].map(new_emotions)
df.head()

df['Emotions'].value_counts()

Emotions
5    5362
0    4666
1    2159
4    1937
2    1304
3     572
Name: count, dtype: int64

In [5]:
# step2:- convert all txt into lower case

df['Text'] = df['Text'].apply(lambda x:x.lower())

In [6]:
# step3:- Remove Puncatuation from text
import string
def remove_punctuation(txt):
    return txt.translate(str.maketrans("","",string.punctuation))

df['Text'] = df['Text'].apply(remove_punctuation)

In [7]:
# step4:- Remove Numbers from text
def remove_numbers(txt):
    new = ''
    for i in txt:
        if not i.isdigit():
            new = new + i
    return new

df['Text'] = df['Text'].apply(remove_numbers)

In [8]:
# step5:- Remove emojis
def remove_emojis(txt):
    clean = ''
    for i in txt:
        if i.isascii():
            clean += i
    return clean

df['Text'] = df['Text'].apply(remove_emojis)


In [9]:
# step6:- Remove Stopwords like is,am etc
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nltk.download('punkt')
nltk.download('stopwords')

stop_words = set(stopwords.words('english'))
stop_words

df.loc[1]['Text'] # 'i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake'

def remove_stopwords(txt):
    words = txt.split()
    cleaned = []
    for i in words:
        if not i in stop_words:
            cleaned.append(i)
    return ' '.join(cleaned)

df['Text'] = df['Text'].apply(remove_stopwords)

df.loc[1]['Text'] # 'go feeling hopeless damned hopeful around someone cares awake'

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\dell\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\dell\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


'go feeling hopeless damned hopeful around someone cares awake'

# Now let's Create model

In [10]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(df['Text'],df['Emotions'],test_size=0.20,random_state=42)

In [11]:
from sklearn.feature_extraction.text import CountVectorizer,TfidfVectorizer

bow_vector = CountVectorizer()
x_train_bow = bow_vector.fit_transform(X_train)
x_test_bow = bow_vector.transform(X_test)

In [12]:
# First Create Naive Bayes model then Creating Logistic Reg Model
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score,classification_report

nb_model = MultinomialNB()
nb_model.fit(x_train_bow,y_train)

nb_pred = nb_model.predict(x_test_bow)
print(accuracy_score(y_test,nb_pred))  # 76 accurracy of Bag of words in Naive Bayes
print(classification_report(y_test,nb_pred))   

0.768125
              precision    recall  f1-score   support

           0       0.75      0.95      0.84       946
           1       0.89      0.63      0.74       427
           2       0.93      0.27      0.41       296
           3       1.00      0.05      0.10       113
           4       0.85      0.57      0.68       397
           5       0.73      0.96      0.83      1021

    accuracy                           0.77      3200
   macro avg       0.86      0.57      0.60      3200
weighted avg       0.80      0.77      0.74      3200



In [13]:
tfidf_vector = TfidfVectorizer()
x_train_tfidf = tfidf_vector.fit_transform(X_train)
x_test_tfidf = tfidf_vector.transform(X_test)

nb2_model = MultinomialNB()
nb2_model.fit(x_train_tfidf,y_train)

nb2_pred = nb2_model.predict(x_test_tfidf)
print(accuracy_score(y_test,nb2_pred)) # 66 accurracy of TF-IDF in Naive Bayes
print(classification_report(y_test,nb2_pred)) 

0.6609375
              precision    recall  f1-score   support

           0       0.70      0.93      0.80       946
           1       0.93      0.29      0.44       427
           2       1.00      0.03      0.06       296
           3       1.00      0.01      0.02       113
           4       0.92      0.22      0.36       397
           5       0.60      0.99      0.74      1021

    accuracy                           0.66      3200
   macro avg       0.86      0.41      0.40      3200
weighted avg       0.76      0.66      0.58      3200



In [14]:
# let's time to do same thing in logistic reg.
from sklearn.linear_model import LogisticRegression

log_model = LogisticRegression()
log_model.fit(x_train_bow,y_train)

log_pred = log_model.predict(x_test_bow)
print(accuracy_score(y_test,log_pred))  # 88 accurracy of Bag of words in LR
print(classification_report(y_test,log_pred))

# ------------------------------------------------
log2_model = LogisticRegression()
log2_model.fit(x_train_tfidf,y_train)

log2_pred = log2_model.predict(x_test_tfidf)
print(accuracy_score(y_test,log2_pred))  # 86 accurracy of TF-IDF in LR
print(classification_report(y_test,log2_pred))


# Isse ye bata chala ke Logistics REgression acchi Accuracy de raha hai as compare to Naive Bayes

0.8896875
              precision    recall  f1-score   support

           0       0.93      0.93      0.93       946
           1       0.89      0.86      0.87       427
           2       0.85      0.76      0.80       296
           3       0.86      0.73      0.79       113
           4       0.86      0.84      0.85       397
           5       0.88      0.94      0.91      1021

    accuracy                           0.89      3200
   macro avg       0.88      0.84      0.86      3200
weighted avg       0.89      0.89      0.89      3200

0.8628125
              precision    recall  f1-score   support

           0       0.90      0.94      0.92       946
           1       0.90      0.81      0.86       427
           2       0.90      0.61      0.73       296
           3       0.88      0.47      0.61       113
           4       0.86      0.76      0.81       397
           5       0.81      0.96      0.88      1021

    accuracy                           0.86      3200
   

In [15]:
import joblib

joblib.dump(log_model, "emotion_model.pkl")
joblib.dump(bow_vector, "bow_vectorizer.pkl")

['bow_vectorizer.pkl']